# Technische Prüfung: Bestand, Restvolumen, Datenlage

Das Gegenstück zu `01_dashboard.ipynb`. Dort steht, was Fachexperten sehen sollen;
hier steht, was zu prüfen ist, bevor man den Zahlen traut – und welche fachlichen
Fragen offen sind.

In [ ]:
# Nur in Google Colab: Projekt aus GitHub installieren, weil das lokale venv dort
# nicht zur Verfuegung steht. Lokal passiert hier nichts, dort liefert `uv sync` die
# Umgebung. Das Repository ist oeffentlich, deshalb braucht pip kein Token.
import importlib

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    # Fuer einen reproduzierbaren Lauf auf einen Tag setzen statt auf "main".
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"

    # Zwei Aufrufe.
    #
    # Der erste beschafft die Abhaengigkeiten, und nur die fehlenden: Colab pinnt
    # pandas 2.2.3 (google-colab) und numpy < 2.3 (numba); "pandas>=2.2" ist damit
    # erfuellt, pip laesst beide stehen. Fuer plotly gilt dasselbe - "plotly>=5" ist
    # von Colabs mitgelieferter Version erfuellt.
    #
    # Der zweite erneuert ausschliesslich unseren Code. --force-reinstall ist noetig,
    # weil die Versionsnummer ueber Commits hinweg 0.1.0 bleibt und pip die
    # Anforderung sonst fuer erfuellt haelt - "pip install git+...@main" laesst einen
    # installierten Stand dann unangetastet, ohne Fehlermeldung (verifiziert am
    # 24.08.2026, der Import schlug danach mit ModuleNotFoundError fehl).
    # --no-deps haelt pandas und numpy aus dem Reinstall heraus: ohne dieses Flag zog
    # der Aufruf pandas 3.0.5 und numpy 2.5.2 nach und brach google-colab 1.0.0 und
    # numba 0.61.2.
    #
    # Nach einem neuen Push zusaetzlich die Runtime neu starten. Sonst bleibt das alte
    # Paket im Speicher, und ein neuer Name in einem alten Modul endet als ImportError.
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
else:
    print("Lokale Installation")

## Parameter und Variablen für die Verarbeitung

In [ ]:
from datetime import date

stichtag = date.today()
abgeschlossene_monate = 12
horizont_monate = 3
projekte_ohne_auftragsvolumen = 20

## Bestand laden

Sechs Abrufe: Kunden, Personen, Sollarbeitszeiten, Projekte, Verbrauch
samt Personenanteilen und Monatsumsätze. Die Eigenheiten der Endpunkte sind in
`umsatzprognose.clockodo` dokumentiert.

In [ ]:
from umsatzprognose import Dashboard

dashboard = Dashboard.laden(
    stichtag=stichtag, abgeschlossene_monate=abgeschlossene_monate, horizont_monate=horizont_monate
)

bestand = dashboard.bestand

print(f"Stichtag: {bestand.stichtag}")
print(f"Projekte gesamt:    {len(bestand.projekte)}")
print(f"davon aktiv:        {len(bestand.aktive_projekte)}")
print(f"davon im Scope:     {len(bestand.im_prognose_scope)}  (aktiv und mit Euro-Budget)")
print(f"Personen:           {len(bestand.mitarbeiter)}")
print(f"Kunden mit Projekt: {len(bestand.kunden)}")

In [ ]:
dashboard.simuliere(monate=horizont_monate)

## Auftragsvolumen und Restvolumen

`roh` ist `Budget − Verbrauch` und kann negativ sein, weil `budget.hard` in dieser
Installation `false` ist. Prognosewirksam ist `max(0, roh)`: eine Überschreitung kann
nur historisch entstehen, die Prognose überschreitet das Budget nicht.

In [ ]:
from umsatzprognose.domaene.zahlen import euro

roh = sum(p.restvolumen_roh or 0.0 for p in bestand.im_prognose_scope)
gesamt = bestand.restvolumen_prognosewirksam

print(f"Auftragsvolumen im Scope:    {euro(bestand.auftragsvolumen):>18}")
print(f"Restvolumen roh:             {euro(roh):>18}")
print(f"Restvolumen prognosewirksam: {euro(gesamt):>18}")
print(f"Summe der Überschreitungen:  {euro(gesamt - roh):>18}")

dashboard.projekttabelle(top=projekte_ohne_auftragsvolumen)

## Aufteilungsschlüssel je Person

Der historische Anteil je Person an den Gesamtstunden des Projekts, aus der
Doppelgruppierung `projects_id` × `users_id`. Er wird unverändert in die Zukunft
fortgeschrieben.

In [ ]:
for projekt in bestand.im_prognose_scope:
    anteile = projekt.anteil_je_mitarbeiter()
    groesste = sorted(anteile.items(), key=lambda paar: paar[1], reverse=True)
    verteilung = ", ".join(f"{person} {anteil:.0%}" for person, anteil in groesste)
    wort = "Person " if len(anteile) == 1 else "Personen"
    print(f"{projekt.bezeichnung[:48]:<48} {len(anteile):>2} {wort} | {verteilung}")

## Sollarbeitszeit

In [ ]:
from umsatzprognose.domaene.zahlen import stunden as stunden_text

aktive = [m for m in bestand.mitarbeiter if m.aktiv]
ohne_sollzeit = [m for m in aktive if m.wochenstunden(bestand.stichtag) is None]
stunden = sum(m.wochenstunden(bestand.stichtag) or 0 for m in aktive)

print(f"Aktive Personen: {len(aktive)}, ohne hinterlegte Sollzeit: {len(ohne_sollzeit)}")
print(f"Vereinbarte Wochenstunden gesamt: {stunden_text(stunden)}")

## Kapazität

Wer im anstehenden Monat noch Kapazität hat, und wie sich die in der Simulation
über den Prognosehorizont tatsächlich verbrauchte Kapazität auf die Projekte im
Scope verteilt - in Personentagen à 7 Stunden.

In [ ]:
dashboard.kapazitaet_je_mitarbeiter(top=100)

In [ ]:
dashboard.kapazitaet_je_projekt(top=1_000)

## Datenlage

In [ ]:
dashboard.hinweise(max_anzahl_betroffen=1_000)

### Aktive Projekte ohne Budget

Sie fallen aus der Prognose, weil ihnen ein bezifferbares Auftragsvolumen fehlt. Ein
Blick auf die Liste unten zeigt, was das überwiegend ist: Schulungs- und
Ausbildungsprodukte aus dem offenen Kursangebot, also Katalogpositionen ohne
beauftragtes Volumen. Zu prüfen bleibt, ob darunter
echte Bestandsprojekte stecken, bei denen nur das Budget fehlt.

In [ ]:
# filter kann hier direkt angepasst werden
projekt_filter = [
    "it-agile GmbH",
    "Öffentliche Schulung",
]

dashboard.projekte_ohne_budget(filter=projekt_filter)

### Aktive Projekte, die als abgeschlossen markiert sind

Sie fallen aus der Prognose. Bitte prüfen!

In [ ]:
beendet = [p for p in bestand.aktive_projekte if p.abgeschlossen]
for projekt in beendet:
    offen = projekt.restvolumen_prognosewirksam
    betrag = "kein Budget" if offen is None else euro(offen)
    print(f"  {projekt.bezeichnung[:58]:<58} offen: {betrag}")